<a href="https://colab.research.google.com/github/Devesh07-OGDemiGOD/Quantum-Walk-Simulation/blob/Quantum-Collab/Task3_LWE_Roll_Number_2025BCY0042.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3 — Learning With Errors (LWE): Roll-Number-Based Demonstration

**Roll number:** `2025BCY0042`  
**Last three digits:** `042` → integer `42`

This notebook is a small educational demonstration of the LWE idea. It uses the required relation

\[
B = (A \times s + e) \bmod q
\]

where `A` is public, `s` is the roll-number-derived secret, `e` is a small error, and `q` is a modulus.

**Note:** The provided Colab reference link could not be inspected directly here, so this notebook is a self-contained implementation of the assignment requirements rather than a line-by-line edit of that exact sample. If your professor expects the sample's exact structure, compare it with the reference notebook before submitting.

## 1. Set the parameters

We use a small modulus `q = 97` for easy demonstration. The secret is computed from the last three roll-number digits modulo `q`. We generate public values `A`, small errors `e`, and calculate `B` using the LWE equation.

In [5]:
import numpy as np
import random

# Roll-number encoding
roll_last_three = 42
q = 97
s = roll_last_three % q

# Reproducibility: remove/change these seeds if you want a new random example
np.random.seed(7)
random.seed(7)

# Public values and small errors (toy parameters for demonstration)
n = 8
A = np.random.randint(0, q, size=n)
e = np.random.randint(-2, 3, size=n)  # small errors: -2, -1, 0, 1, 2

# LWE relation, applied element-by-element
B = (A * s + e) % q

print("STEP 1 — Parameters")
print(f"Roll-number suffix = {roll_last_three:03d}")
print(f"Secret key s = {roll_last_three} mod {q} = {s}")
print(f"Modulus q = {q}")
print()
print("STEP 2 — Public values A")
print("A =", A)
print()
print("STEP 3 — Small error values e")
print("e =", e)
print()
print("STEP 4 — Calculate B = (A*s + e) mod q")
print("B =", B)

STEP 1 — Parameters
Roll-number suffix = 042
Secret key s = 42 mod 97 = 42
Modulus q = 97

STEP 2 — Public values A
A = [47 68 25 67 83 23 92 57]

STEP 3 — Small error values e
e = [-2 -1  0  0 -2  2 -2  2]

STEP 4 — Calculate B = (A*s + e) mod q
B = [32 42 80  1 89 95 79 68]


## 2. Randomly choose a message bit and encrypt it

The message bit is chosen randomly as `0` or `1`. For this toy demonstration, a random binary vector `r` selects/combine entries from the public values. The ciphertext is `(u, v)`, where the message is encoded by adding approximately `q/2` when the bit is `1`.

This is an **illustrative toy scheme**, not secure production cryptography.

In [6]:
message = random.randint(0, 1)
r = np.random.randint(0, 2, size=n)  # random binary selection vector

# Toy encryption
u = int(np.dot(r, A) % q)
v = int((np.dot(r, B) + message * (q // 2)) % q)

print("STEP 5 — Choose a random message bit")
print("message =", message)
print("Random binary vector r =", r)
print()
print("STEP 6 — Encrypt the message")
print("u = (r · A) mod q =", u)
print("v = (r · B + message * floor(q/2)) mod q =", v)
print("Ciphertext (u, v) =", (u, v))

STEP 5 — Choose a random message bit
message = 1
Random binary vector r = [0 1 1 0 0 1 0 0]

STEP 6 — Encrypt the message
u = (r · A) mod q = 19
v = (r · B + message * floor(q/2)) mod q = 71
Ciphertext (u, v) = (19, 71)


## 3. Demonstrate recovery with the secret key

Using the secret `s`, calculate the centered value of `v - u*s (mod q)`. Values near `0` represent bit `0`; values near `q/2` represent bit `1`. This works as a teaching demonstration because the errors are deliberately small and the parameters are tiny.

In [7]:
raw = int((v - u * s) % q)
# Center the residue so values above q/2 are interpreted as negative
centered = raw if raw <= q // 2 else raw - q
decoded_message = 0 if abs(centered) < q / 4 else 1

print("STEP 7 — Decrypt using the secret key")
print("Raw residue (v - u*s) mod q =", raw)
print("Centered residue =", centered)
print("Original message =", message)
print("Decoded message =", decoded_message)
print("Decryption successful? ", decoded_message == message)

STEP 7 — Decrypt using the secret key
Raw residue (v - u*s) mod q = 49
Centered residue = -48
Original message = 1
Decoded message = 1
Decryption successful?  True


## 4. Short questions

### Q1. What happens to B when you change the secret key s or the message bit?
Changing `s` generally changes `B`, because `B = (A*s + e) mod q`. The message bit does not change this public-key vector `B` in this demonstration; it changes the ciphertext component `v`, where the bit is encoded.

### Q2. Why is it hard to find s if you only know A and B?
The small error vector `e` hides the exact relationship between `A`, `s`, and `B`. Recovering `s` from many noisy modular equations is the Learning With Errors problem; real systems use much larger, carefully chosen parameters, unlike this tiny classroom example.

### Q3. How does this example show the idea of quantum-safe encryption?
LWE-based cryptography is considered a leading approach to post-quantum cryptography because its security is based on hard mathematical problems for which no efficient quantum attack is known that breaks properly parameterized schemes. This toy code only illustrates the concept; it is **not secure encryption** and should not be used to protect real information.

## 5. Optional experiment: change the secret key

Run this cell to compare the original `B` with the values obtained using a different secret. It leaves the original `A`, `e`, `s`, and `B` unchanged.

In [8]:
alternative_s = (s + 1) % q
B_with_changed_s = (A * alternative_s + e) % q

print("Original secret s =", s)
print("Alternative secret s =", alternative_s)
print("Original B =", B)
print("B with changed s =", B_with_changed_s)
print("Are the B vectors identical?", np.array_equal(B, B_with_changed_s))

Original secret s = 42
Alternative secret s = 43
Original B = [32 42 80  1 89 95 79 68]
B with changed s = [79 13  8 68 75 21 74 28]
Are the B vectors identical? False
